In [25]:
%pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.4 MB/s eta 0:00:0000:01


In [26]:
from google.colab import drive
import os

# Grab the paths to the metadata files and the waves file from google drive
drive.mount("/content/drive")
BASE_DIR = "/content/drive/My Drive/Computer Science/Jabbel/data"
csv_path = os.path.join(BASE_DIR, "metadata.csv")
audio_dir = os.path.join(BASE_DIR, "wavs")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load Dataset
```python
datasets.load_dataset()
```
This function is from HuggingFace that loads in audio dir based on the file names e.g (metedata.csv and wavs dir)

In [27]:
import datasets

dataset = datasets.load_dataset(
    "csv",
    data_files = csv_path,
    # Tells HF the divider that seperates the ID : Transcript... (normally a , but it is a | for me) 
    sep = "|",                                      
     # Tells PyTorch the headers ID : Transcript : Norm_Trans for each record in the csv, as there's no headers
    column_names=["file_id", "transcription", "normalized_transcription"],   
    split = "train"                 
)

# Do not need the normal one as the normalized one will be used for training
dataset = dataset.remove_columns(["transcription"])

# Create Full Paths to .wav files
Because the csv file only lists the ID and not the file extention, I need to map over the csv and manually add them so the ful path is built
1. <code>batch</code>: This takes mutiple rows at a time, instead of looking at one row at a time.
2. <code>f"{fid}.wav"</code>: This takes the ID and ads .wav onto the end of each ID
3. <code>batch["audio"]</code>: This adds a new column called audio with all the correct path names
4. <code>dataset.map</code>: This tells the dataset to run the function across every single row in the dataset


In [28]:
def add_audio_paths(batch):
    batch["audio"] = [os.path.join(audio_dir, f"{fid}.wav") for fid in batch["file_id"]]
    return batch

dataset = dataset.map(add_audio_paths, batched = True)

# Cast The Path Strings To Actual Audio Files
1. Makes the audio column, currently just text filenames, be treatd as actual audio rather than just plane text
2. <code>Audio(sampling_rate = 16_000)</code>: Tells the model no matter what make alll these files 16,000 Hz to make sure they are all the same Hz

In [29]:
dataset = dataset.cast_column("audio", datasets.Audio(sampling_rate = 16_000))

print(dataset[0]["audio"])
print(dataset[0])

{'file_id': 'LJ001-0001', 'normalized_transcription': 'Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7bf368f94440>}


# Cleaning Dataset of Accent Chars
In the readme file it says that 19 files have special non ASCHII chars, therefore i need to clean them and just turn them into non accent chars just so the tokenization works

In [ ]:
from unidecode import unidecode 

def clean_text(batch):
    batch["normalized_transcription"] = [unidecode(t) for t in batch["normalized_transcription"]]
    return batch

dataset = dataset.map(clean_text, batched = True)

# Creating Tokenization Function

In [33]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders

TOKENIZER_PATH = os.path.join(BASE_DIR, "tokenizer.json")

def get_tokenizer(save_path = TOKENIZER_PATH):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.add_special_tokens(["[PAD]", "[UNK]", " "])
    tokenizer.add_tokens(list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz' "))

    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space = True)
    tokenizer.decoder = decoders.ByteLevel()
    tokenizer.save(save_path)
    return tokenizer


In [34]:
tokenizer = get_tokenizer()
tokenizer.get_vocab()

{'F': 8,
 't': 48,
 'R': 20,
 'I': 11,
 'D': 6,
 'B': 4,
 'g': 35,
 'H': 10,
 'E': 7,
 'A': 3,
 'b': 30,
 'c': 31,
 'n': 42,
 'w': 51,
 's': 47,
 "'": 55,
 'N': 16,
 'o': 43,
 'y': 53,
 'l': 40,
 'f': 34,
 'O': 17,
 'd': 32,
 '[PAD]': 0,
 'k': 39,
 'm': 41,
 'x': 52,
 'Z': 28,
 'X': 26,
 'L': 14,
 'V': 24,
 'S': 21,
 'K': 13,
 'Q': 19,
 'W': 25,
 'M': 15,
 'u': 49,
 'z': 54,
 ' ': 2,
 'j': 38,
 'v': 50,
 'p': 44,
 'G': 9,
 'q': 45,
 'P': 18,
 'T': 22,
 '[UNK]': 1,
 'r': 46,
 'Y': 27,
 'J': 12,
 'h': 36,
 'U': 23,
 'e': 33,
 'a': 29,
 'C': 5,
 'i': 37}

# Creating Wrapper For Dataset
This is where i create the brdige between the Hugging Face dataset and the tokenizer objects and how PyTorch can understand the dataset 

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# The Wrapper Class
class AudioDataset(Dataset):
    def __init__(self, dataset, tokenizer = None, num_samples = None):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.num_samples = len(dataset) if num_samples is None else num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        item = self.dataset[idx]

        # Convert the waveform to a torch tensor
        waveform = torch.from_numpy(item["audio"]["array"]).float()

        text = item["normalized_transcription"]

        if self.tokenizer:
            encoded = self.tokenizer.encode(text)
            return {"audio": waveform,"text": text ,"input_ids": encoded.ids}
